In [58]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2


In [59]:
PREDICTORS = ["pH", "Cond", "Temp", "OD", "Tds", "ORP"] # 6 entradas
TARGETS = ["Fe", "Al", "As", "Pb", "Zn", "Hg", "Co", "V", "Ba", "Mn"] # 10 saidas

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

N_COMPONENTS = 4

In [60]:
Dataset = pd.read_excel("../Dados/Dados.xlsx")

Datasets = []

for n in range(1, 5):
    n_data = Dataset[ Dataset["Pontos"] == f"P{n}"]
    n_data = n_data.drop(columns=["Campanhas", "Pontos"])
    
    n_data_norm = n_data
        
    Datasets.append(n_data)

# RNA

In [ ]:
def PrepareData(VirtualDataset, OriginalDataset, target, i):
    X_orig = OriginalDataset[PREDICTORS].values
    Y_orig = OriginalDataset[target].values
    
    Xv = VirtualDataset[PREDICTORS].values
    Yv = VirtualDataset[target].values
    
    X_train, X_test, Y_train, Y_test = train_test_split(
        X_orig, Y_orig, test_size=0.2, random_state=42
    )
    
    # CONCATENAÇÃO CORRETA
    X_train = np.concatenate((X_train, Xv), axis=0)
    Y_train = np.concatenate((Y_train, Yv), axis=0)

    x_train = SCALER.fit_transform(X_train)
    x_test  = SCALER.transform(X_test)
    
    return x_train, x_test, Y_train, Y_test

    
def PrintDim(x, y):
    print(f"Dimensão da entrada: {np.shape(x)}")
    print(f"Dimensão da saida: {np.shape(y)}")

In [63]:
from sklearn.metrics import mean_squared_error, r2_score

def TrainANN(
    x_train, y_train,
    x_test, y_test,
    n_hidden=[],
    lr=1e-3,
    l2_reg=1e-4,
    epochs=500,
    batch_size=8,
    verbose=0
):
    n_inputs = x_train.shape[1]

    # ======================
    # Modelo
    # ======================
    model = Sequential()

    model.add(
        Dense(
            n_hidden[0],
            activation="relu",
            kernel_regularizer=l2(l2_reg),
            input_shape=(n_inputs,)
        )
    )

    for units in n_hidden[1:]:
        model.add(
            Dense(
                units,
                activation="relu",
                kernel_regularizer=l2(l2_reg)
            )
        )

    model.add(Dense(1, activation="linear"))

    w0 = model.get_weights()


    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss="mse"
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=30,
        restore_best_weights=True
    )

    # ======================
    # Treinamento
    # ======================
    history = model.fit(
        x_train, y_train,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=verbose
    )
    wf = model.get_weights()

    # ======================
    # Predições (NORMALIZADAS)
    # ======================
    y_train_pred_norm = model.predict(x_train, verbose=0)
    y_test_pred_norm  = model.predict(x_test,  verbose=0)

    # ======================
    # DESNORMALIZAÇÃO
    # ======================
    y_train_real = OUT_SCALER.inverse_transform(y_train.reshape(-1, 1)).ravel()
    y_test_real  = OUT_SCALER.inverse_transform(y_test.reshape(-1, 1)).ravel()

    y_train_pred = OUT_SCALER.inverse_transform(y_train_pred_norm).ravel()
    y_test_pred  = OUT_SCALER.inverse_transform(y_test_pred_norm).ravel()

    # ======================
    # MÉTRICAS NO ESPAÇO FÍSICO
    # ======================
    metrics = {
    "mse_train": round(mean_squared_error(y_train_real, y_train_pred), 4),
    "mse_test":  round(mean_squared_error(y_test_real,  y_test_pred), 4),
    "r2_train":  round(r2_score(y_train_real, y_train_pred), 4),
    "r2_test":   round(r2_score(y_test_real,  y_test_pred), 4)
}


    return model, history, metrics, w0, wf


In [64]:
neurons = [[1], [2], [4], [6], [8], [10], [14], [16], [18], [20]]
all_metrics = []

for i, Dataset in enumerate(Datasets):
    os.makedirs(f"./Dados/VirtualData/P{i+1}/FilterResults/", exist_ok=True)

    for j, target in enumerate(TARGETS):
        
        print(f" → {target}")
        output_dir = f"./Dados/VirtualData/P{i+1}"
        vs_filename = os.path.join(output_dir, f"virtual_samples_{target}.xlsx")
        VirtualDataset = pd.read_excel(vs_filename, sheet_name="orig-vs")
        x_train, x_test, y_train, y_test = PrepareData(VirtualDataset, Dataset, target, i)
        
        y_train = OUT_SCALER.fit_transform(y_train.reshape(-1, 1)).ravel()
        y_test  = OUT_SCALER.transform(y_test.reshape(-1, 1)).ravel()
        PrintDim(x_train, y_train)
        PrintDim(x_test, y_test)
        
        for neuron in neurons:
            for k in range(10):
                model, history, metrics, w0, wf = TrainANN(
                    x_train, y_train,
                    x_test, y_test,
                    n_hidden=neuron
                )

                # Apenas adiciona metadados
                metrics.update({
                    "P": i + 1,
                    "runTime": k,
                    "target": target,
                    "neurons": neuron,
                    "W0": str([w.round(4).tolist() for w in w0]),
                    "Wf": str([w.round(4).tolist() for w in wf]),
                })

                all_metrics.append(metrics)
                print(metrics)
                # break
        break
    break

results = pd.DataFrame(all_metrics)
print(results)

 → Fe
Dimensão da entrada: (104, 6)
Dimensão da saida: (104,)
Dimensão da entrada: (6, 6)
Dimensão da saida: (6,)
{'mse_train': 10910.9079, 'mse_test': 28407.8143, 'r2_train': 0.8558, 'r2_test': 0.8007, 'P': 1, 'runTime': 0, 'target': 'Fe', 'neurons': [1], 'W0': '[[[0.8608999848365784], [-0.2540000081062317], [-0.017000000923871994], [-0.1265999972820282], [-0.0794999971985817], [0.17030000686645508]], [0.0], [[-0.412200003862381]], [0.0]]', 'Wf': '[[[-0.027400000020861626], [0.09470000118017197], [-0.8960000276565552], [0.3467999994754791], [0.25440001487731934], [-0.015799999237060547]], [1.1991000175476074], [[-1.0611000061035156]], [1.315500020980835]]'}
{'mse_train': 12181.8351, 'mse_test': 31161.4851, 'r2_train': 0.839, 'r2_test': 0.7813, 'P': 1, 'runTime': 1, 'target': 'Fe', 'neurons': [1], 'W0': '[[[0.46050000190734863], [0.2736000120639801], [-0.155799999833107], [-0.8402000069618225], [-0.6597999930381775], [-0.9009000062942505]], [0.0], [[-0.20960000157356262]], [0.0]]', 'Wf

In [1]:
with pd.ExcelWriter("Results.xlsx", engine="openpyxl") as writer:
            results.to_excel(writer, index=False, sheet_name="Vs")

NameError: name 'pd' is not defined